<a href="https://colab.research.google.com/github/katheleya/-sort-names-cpp/blob/main/Copy_of_Lab1_99383.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 1:Association Rule Mining Market Basket Analysis**

Objective: Use Apriori and FP-Growth to find products that occur together in grocery baskets, then interpret support, confidence and lift.
- Part A: The six transactions supplied in the starter notebook.
- Part B: A new, manually constructed dataset containing 20 transactions.
- Baseline settings: Minimum support = 0.5; minimum confidence = 0.7.
- Extension: Compare Apriori with FP-Growth and change the thresholds.
Run the code cells from top to bottom. The last Markdown cell contains the written answers to all three lab analysis questions. The data are synthetic and do not describe a real store.

**Import required libraries**


*   Run the installation cell first. This notebook uses mlxtend==0.23.4 for reproducibility.
*   Pandas is installed as a dependency of mlxtend.

In [ ]:
%pip install -q "mlxtend==0.23.4"

In [ ]:
import pandas as pd
import mlxtend
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.4f}".format)

print("mlxtend version:", mlxtend.__version__)
print("pandas version:", pd.__version__)

mlxtend version: 0.23.4
pandas version: 2.2.3


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


These two small functions make the output easier to read. They turn itemsets into readable names and sort rules by lift, followed by confidence and support. They do not change the calculated values.

In [ ]:
def itemset_table(itemsets):
    table = itemsets.copy()
    table["itemsets"] = table["itemsets"].apply(
        lambda items: ", ".join(sorted(items))
    )
    return table.sort_values(
        ["support", "itemsets"], ascending=[False, True]
    ).reset_index(drop=True)


def rule_table(rules):
    columns = ["antecedents", "consequents", "support", "confidence", "lift"]
    table = rules[columns].copy()
    for column in ["antecedents", "consequents"]:
        table[column] = table[column].apply(
            lambda items: ", ".join(sorted(items))
        )
    return table.sort_values(
        ["lift", "confidence", "support", "antecedents", "consequents"],
        ascending=[False, False, False, True, True]
    ).reset_index(drop=True)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Step 1: Create and encode the original dataset**

Each list is one shopping transaction. One-hot encoding creates one column per item: True means that the item is present and False means that it is absent. Keep the Boolean DataFrame for mining.

In [ ]:
transactions = [
    ["Bread", "Milk", "Eggs"],  # Transaction 1
    ["Bread", "Diapers", "Beer", "Eggs"],  # Transaction 2
    ["Milk", "Diapers", "Beer", "Cola"],  # Transaction 3
    ["Bread", "Milk", "Diapers", "Beer"],  # Transaction 4
    ["Bread", "Milk", "Diapers", "Cola"],  # Transaction 5
    ["Bread", "Milk", "Eggs", "Beer"],  # Transaction 6
]

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("Number of transactions:", len(df))
display(df)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of transactions: 6


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Beer,Bread,Cola,Diapers,Eggs,Milk
0,False,True,False,False,True,True
1,True,True,False,True,True,False
2,True,False,True,True,False,True
3,True,True,False,True,False,True
4,False,True,True,True,False,True
5,True,True,False,False,True,True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Step 2: Apply Apriori**

With six transactions, min_support=0.5 keeps an itemset if all its items appear together in at least 3 transactions. use_colnames=True returns item names instead of column numbers.

In [ ]:
frequent_itemsets = apriori(df, min_support=0.5, use_colnames=True)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(itemset_table(frequent_itemsets))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of frequent itemsets: 12


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
0,0.8333,Bread
1,0.8333,Milk
2,0.6667,Beer
3,0.6667,"Bread, Milk"
4,0.6667,Diapers
5,0.5000,"Beer, Bread"
6,0.5000,"Beer, Diapers"
7,0.5000,"Beer, Milk"
8,0.5000,"Bread, Diapers"
9,0.5000,"Bread, Eggs"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Step 3: Generate association rules**

Keep rules with confidence of at least 0.7, then display them in descending order of lift. num_itemsets=len(df) supplies the number of original transactions. We do not filter out rules with lift below 1 because the lab asks us to inspect the confidence-filtered results.

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.7,
    num_itemsets=len(df)
)

final_rules = rule_table(rules)
print("Number of rules:", len(final_rules))
display(final_rules)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of rules: 9


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,support,confidence,lift
0,Eggs,Bread,0.5000,1.0000,1.2000
1,Beer,Diapers,0.5000,0.7500,1.1250
2,Diapers,Beer,0.5000,0.7500,1.1250
3,Bread,Milk,0.6667,0.8000,0.9600
4,Milk,Bread,0.6667,0.8000,0.9600
5,Beer,Bread,0.5000,0.7500,0.9000
6,Beer,Milk,0.5000,0.7500,0.9000
7,Diapers,Bread,0.5000,0.7500,0.9000
8,Diapers,Milk,0.5000,0.7500,0.9000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Step 4: Create a new dataset with 20 transactions**

The new dataset contains Bread, Milk, Butter, Eggs, Coffee and Rice. Similar baskets can occur in different shopping visits, so repeated baskets are kept as separate transactions.

In [ ]:
my_transactions = [
    ["Bread", "Milk", "Butter"],  # Transaction 1
    ["Bread", "Milk", "Butter", "Eggs"],  # Transaction 2
    ["Bread", "Milk", "Butter", "Coffee"],  # Transaction 3
    ["Bread", "Milk", "Butter"],  # Transaction 4
    ["Bread", "Milk", "Butter", "Eggs"],  # Transaction 5
    ["Bread", "Milk", "Butter", "Coffee"],  # Transaction 6
    ["Bread", "Milk", "Butter"],  # Transaction 7
    ["Bread", "Milk", "Butter", "Eggs"],  # Transaction 8
    ["Bread", "Milk", "Butter", "Coffee"],  # Transaction 9
    ["Bread", "Butter", "Eggs"],  # Transaction 10
    ["Bread", "Butter", "Coffee"],  # Transaction 11
    ["Milk", "Butter", "Eggs"],  # Transaction 12
    ["Bread", "Milk", "Eggs"],  # Transaction 13
    ["Bread", "Milk", "Coffee"],  # Transaction 14
    ["Bread", "Eggs"],  # Transaction 15
    ["Milk", "Rice"],  # Transaction 16
    ["Milk", "Coffee"],  # Transaction 17
    ["Milk", "Eggs"],  # Transaction 18
    ["Rice", "Coffee"],  # Transaction 19
    ["Rice", "Eggs"],  # Transaction 20
]

print("Number of transactions:", len(my_transactions))
display(pd.DataFrame({
    "transaction_id": range(1, len(my_transactions) + 1),
    "items": [", ".join(items) for items in my_transactions]
}))

my_te = TransactionEncoder()
my_encoded = my_te.fit(my_transactions).transform(my_transactions)
my_df = pd.DataFrame(my_encoded, columns=my_te.columns_)
display(my_df)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of transactions: 20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,transaction_id,items
0,1,"Bread, Milk, Butter"
1,2,"Bread, Milk, Butter, Eggs"
2,3,"Bread, Milk, Butter, Coffee"
3,4,"Bread, Milk, Butter"
4,5,"Bread, Milk, Butter, Eggs"
5,6,"Bread, Milk, Butter, Coffee"
6,7,"Bread, Milk, Butter"
7,8,"Bread, Milk, Butter, Eggs"
8,9,"Bread, Milk, Butter, Coffee"
9,10,"Bread, Butter, Eggs"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Bread,Butter,Coffee,Eggs,Milk,Rice
0,True,True,False,False,True,False
1,True,True,False,True,True,False
2,True,True,True,False,True,False
3,True,True,False,False,True,False
4,True,True,False,True,True,False
5,True,True,True,False,True,False
6,True,True,False,False,True,False
7,True,True,False,True,True,False
8,True,True,True,False,True,False
9,True,True,False,True,False,False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
my_counts = pd.DataFrame({
    "count": my_df.sum(),
    "support": my_df.mean()
})
display(my_counts)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,count,support
Bread,14,0.7000
Butter,12,0.6000
Coffee,7,0.3500
Eggs,9,0.4500
Milk,15,0.7500
Rice,3,0.1500


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Repeat Step 2: Apriori on the new dataset**

A support threshold of 0.5 now requires an itemset to appear in at least 10 of the 20 transactions.

In [ ]:
my_frequent_itemsets = apriori(
    my_df, min_support=0.5, use_colnames=True
)

print("Number of frequent itemsets:", len(my_frequent_itemsets))
display(itemset_table(my_frequent_itemsets))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of frequent itemsets: 6


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,support,itemsets
0,0.7500,Milk
1,0.7000,Bread
2,0.6000,Butter
3,0.5500,"Bread, Butter"
4,0.5500,"Bread, Milk"
5,0.5000,"Butter, Milk"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Repeat Step 3: Association rules on the new dataset**

In [ ]:
my_rules = association_rules(
    my_frequent_itemsets,
    metric="confidence",
    min_threshold=0.7,
    num_itemsets=len(my_df)
)

my_final_rules = rule_table(my_rules)
print("Number of rules:", len(my_final_rules))
display(my_final_rules)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Number of rules: 5


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
0,Butter,Bread,0.5500,0.9167,1.3095
1,Bread,Butter,0.5500,0.7857,1.3095
2,Butter,Milk,0.5000,0.8333,1.1111
3,Bread,Milk,0.5500,0.7857,1.0476
4,Milk,Bread,0.5500,0.7333,1.0476


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Extension 1: Use FP-Growth**

Use the same 20 transactions, support threshold and confidence threshold so that the algorithm comparison is fair.

In [ ]:
fp_itemsets = fpgrowth(my_df, min_support=0.5, use_colnames=True)

fp_rules = association_rules(
    fp_itemsets,
    metric="confidence",
    min_threshold=0.7,
    num_itemsets=len(my_df)
)

print("FP-Growth frequent itemsets:", len(fp_itemsets))
display(itemset_table(fp_itemsets))

print("FP-Growth rules:", len(fp_rules))
display(rule_table(fp_rules))

# Compare the complete outputs after putting rows into the same order.
pd.testing.assert_frame_equal(
    itemset_table(my_frequent_itemsets), itemset_table(fp_itemsets)
)
pd.testing.assert_frame_equal(my_final_rules, rule_table(fp_rules))
print("Apriori and FP-Growth return the same itemsets and rule metrics.")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

FP-Growth frequent itemsets: 6


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,support,itemsets
0,0.7500,Milk
1,0.7000,Bread
2,0.6000,Butter
3,0.5500,"Bread, Butter"
4,0.5500,"Bread, Milk"
5,0.5000,"Butter, Milk"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

FP-Growth rules: 5


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
0,Butter,Bread,0.5500,0.9167,1.3095
1,Bread,Butter,0.5500,0.7857,1.3095
2,Butter,Milk,0.5000,0.8333,1.1111
3,Bread,Milk,0.5500,0.7857,1.0476
4,Milk,Bread,0.5500,0.7333,1.0476


Apriori and FP-Growth return the same itemsets and rule metrics.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Extension 2: Change the thresholds**

The first five scenarios change one threshold at a time so that its effect is clear. The final scenario lowers both thresholds. Every scenario uses the same 20 transactions and is tested with both algorithms.

The code also handles an empty frequent-itemset result. A nonempty itemset table containing only single items can still produce zero rules.

In [ ]:
scenarios = [
    ("Baseline",          0.5, 0.7),
    ("Lower support",     0.3, 0.7),
    ("Higher support",    0.6, 0.7),
    ("Lower confidence",  0.5, 0.5),
    ("Higher confidence", 0.5, 0.9),
    ("Both lower",        0.3, 0.5)
]

experiment_results = []
experiment_rule_tables = {}

for scenario, min_support, min_confidence in scenarios:
    for algorithm_name, algorithm in [
        ("Apriori", apriori), ("FP-Growth", fpgrowth)
    ]:
        itemsets = algorithm(
            my_df, min_support=min_support, use_colnames=True
        )

        if itemsets.empty:
            current_rules = pd.DataFrame(columns=[
                "antecedents", "consequents", "support", "confidence", "lift"
            ])
        else:
            current_rules = association_rules(
                itemsets,
                metric="confidence",
                min_threshold=min_confidence,
                num_itemsets=len(my_df)
            )

        experiment_results.append({
            "Scenario": scenario,
            "Algorithm": algorithm_name,
            "Min support": min_support,
            "Min confidence": min_confidence,
            "Frequent itemsets": len(itemsets),
            "Rules": len(current_rules)
        })
        experiment_rule_tables[(scenario, algorithm_name)] = rule_table(
            current_rules
        )

comparison = pd.DataFrame(experiment_results)
display(comparison)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Scenario,Algorithm,Min support,Min confidence,Frequent itemsets,Rules
0,Baseline,Apriori,0.5000,0.7000,6,5
1,Baseline,FP-Growth,0.5000,0.7000,6,5
2,Lower support,Apriori,0.3000,0.7000,11,9
3,Lower support,FP-Growth,0.3000,0.7000,11,9
4,Higher support,Apriori,0.6000,0.7000,3,0
5,Higher support,FP-Growth,0.6000,0.7000,3,0
6,Lower confidence,Apriori,0.5000,0.5000,6,6
7,Lower confidence,FP-Growth,0.5000,0.5000,6,6
8,Higher confidence,Apriori,0.5000,0.9000,6,1
9,Higher confidence,FP-Growth,0.5000,0.9000,6,1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
# Inspect actual rules from three changed settings.
print("Lower support (0.3), confidence unchanged (0.7):")
display(experiment_rule_tables[("Lower support", "Apriori")])

print("Lower confidence (0.5), support unchanged (0.5):")
display(experiment_rule_tables[("Lower confidence", "Apriori")])

print("Higher confidence (0.9), support unchanged (0.5):")
display(experiment_rule_tables[("Higher confidence", "Apriori")])

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Lower support (0.3), confidence unchanged (0.7):


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
0,"Bread, Milk",Butter,0.4500,0.8182,1.3636
1,Butter,"Bread, Milk",0.4500,0.7500,1.3636
2,Butter,Bread,0.5500,0.9167,1.3095
3,Bread,Butter,0.5500,0.7857,1.3095
4,"Butter, Milk",Bread,0.4500,0.9000,1.2857
5,Butter,Milk,0.5000,0.8333,1.1111
6,"Bread, Butter",Milk,0.4500,0.8182,1.0909
7,Bread,Milk,0.5500,0.7857,1.0476
8,Milk,Bread,0.5500,0.7333,1.0476


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Higher confidence (0.9), support unchanged (0.5):


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
0,Butter,Bread,0.5500,0.9167,1.3095


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**References**

The following official mlxtend documentation was used to check the API and metric definitions:
- Association rules: support, confidence, lift and rule generation
- FP-Growth: frequent itemsets and comparison with Apriori
The calculations and interpretations below refer to the actual outputs of this notebook.

## Lab Deliverables and Analysis

### 1. Interpretation of the Original Dataset

The original dataset contains six grocery transactions. Using a minimum support of **0.5** and a minimum confidence of **0.7**, Apriori produces **12 frequent itemsets and 9 association rules**.

#### Meaning of Support, Confidence and Lift

Let **A** represent the antecedent, **B** represent the consequent, and **N** represent the total number of transactions.

**Support** measures the proportion of all transactions containing every item in both A and B.

$$
\mathrm{Support}(A \rightarrow B)
=
\frac{\mathrm{Count}(A \cup B)}{N}
$$

**Confidence** measures how often B appears in transactions that contain A.

$$
\mathrm{Confidence}(A \rightarrow B)
=
\frac{\mathrm{Count}(A \cup B)}{\mathrm{Count}(A)}
$$

**Lift** compares the observed association with what would be expected if A and B were independent.

$$
\mathrm{Lift}(A \rightarrow B)
=
\frac{\mathrm{Confidence}(A \rightarrow B)}{\mathrm{Support}(B)}
$$

- **Lift > 1:** Positive association.
- **Lift = 1:** Co-occurrence matches the level expected under independence.
- **Lift < 1:** Negative association in the dataset.

These measures describe associations. They do not prove that buying one item causes someone to buy another.

#### Results Sorted by Lift

| Rule | Support | Confidence | Lift |
|:---|---:|---:|---:|
| Eggs → Bread | 0.5000 | 1.0000 | 1.2000 |
| Beer → Diapers | 0.5000 | 0.7500 | 1.1250 |
| Diapers → Beer | 0.5000 | 0.7500 | 1.1250 |
| Bread → Milk | 0.6667 | 0.8000 | 0.9600 |
| Milk → Bread | 0.6667 | 0.8000 | 0.9600 |
| Beer → Bread | 0.5000 | 0.7500 | 0.9000 |
| Beer → Milk | 0.5000 | 0.7500 | 0.9000 |
| Diapers → Bread | 0.5000 | 0.7500 | 0.9000 |
| Diapers → Milk | 0.5000 | 0.7500 | 0.9000 |

#### Interpretation of the Rules

**Eggs → Bread**

This rule has the highest lift in the original dataset.

- **Support = 0.5000:** Eggs and Bread appear together in 3 of the 6 transactions.
- **Confidence = 1.0000:** All 3 transactions containing Eggs also contain Bread.
- **Lift = 1.2000:** Bread appears in 5 of the 6 transactions overall. Therefore, lift is `1 ÷ (5/6) = 1.2`, indicating a positive association.

The reverse rule, Bread → Eggs, has confidence `3/5 = 0.6`. It does not meet the minimum confidence of 0.7.

**Beer → Diapers**

- **Support = 0.5000:** Beer and Diapers appear together in 3 of the 6 transactions.
- **Confidence = 0.7500:** Of the 4 transactions containing Beer, 3 also contain Diapers.
- **Lift = 1.1250:** Diapers appears in 4 of the 6 transactions overall. The lift is `0.75 ÷ (4/6) = 1.125`, indicating a positive association.

Diapers → Beer has the same values because Beer and Diapers each appear in 4 transactions.

**Bread → Milk**

- **Support = 0.6667:** Bread and Milk appear together in 4 of the 6 transactions.
- **Confidence = 0.8000:** Of the 5 transactions containing Bread, 4 also contain Milk.
- **Lift = 0.9600:** Milk already appears in 5 of the 6 transactions overall. The lift is `0.8 ÷ (5/6) = 0.96`.

Although confidence is 80%, the lift is below 1. Therefore, this rule does not show a positive association. Milk → Bread has the same values.

**Remaining Rules**

Beer → Bread, Beer → Milk, Diapers → Bread and Diapers → Milk each have support **0.5**, confidence **0.75** and lift **0.9**. They meet the confidence threshold, but their lift values indicate negative association in this dataset.

Only **3 of the 9 rules** have lift above 1.

**Why Some Patterns Are Missing**

Cola appears in only 2 of the 6 transactions. Its support is `2/6 = 0.3333`, which is below 0.5.

The example rule Bread and Milk → Eggs also does not qualify. Its support is `2/6 = 0.3333`, and its confidence is `2/4 = 0.5`.

---

### 2. Interpretation of the New 20-Transaction Dataset

I created a synthetic dataset containing 20 grocery transactions using Bread, Milk, Butter, Eggs, Coffee and Rice. Each basket represents a separate shopping visit, so repeated baskets are counted as separate transactions.

Using minimum support **0.5** and minimum confidence **0.7**, Apriori produces **6 frequent itemsets and 5 association rules**.

#### Results Sorted by Lift

| Rule | Support | Confidence | Lift |
|:---|---:|---:|---:|
| Butter → Bread | 0.5500 | 0.9167 | 1.3095 |
| Bread → Butter | 0.5500 | 0.7857 | 1.3095 |
| Butter → Milk | 0.5000 | 0.8333 | 1.1111 |
| Bread → Milk | 0.5500 | 0.7857 | 1.0476 |
| Milk → Bread | 0.5500 | 0.7333 | 1.0476 |

#### Interpretation of the Rules

**Butter → Bread**

Butter appears in 12 transactions, Bread appears in 14 transactions, and both appear together in 11 transactions.

- **Support:** `11/20 = 0.55`, meaning 55% of all transactions contain both products.
- **Confidence:** `11/12 = 0.9167`, meaning 91.67% of transactions containing Butter also contain Bread.
- **Lift:** `(11/12) ÷ (14/20) = 1.3095`.

Bread is approximately **1.31 times as likely** in a transaction containing Butter as in a transaction overall.

Under independence, the expected joint support would be `0.60 × 0.70 = 0.42`. The observed joint support is higher at **0.55**, indicating positive association.

**Bread → Butter**

The support remains **0.55**, but the confidence is `11/14 = 0.7857`, or **78.57%**. The lift remains **1.3095**.

This shows that reversing a rule can change its confidence while support and lift remain the same. Butter → Bread and Bread → Butter tie for the highest lift.

**Butter → Milk**

Butter and Milk appear together in 10 transactions.

- **Support:** `10/20 = 0.50`.
- **Confidence:** `10/12 = 0.8333`, or 83.33%.
- **Lift:** `(10/12) ÷ (15/20) = 1.1111`.

The lift is above 1, indicating positive association.

**Bread → Milk and Milk → Bread**

Both rules have support **0.55** because Bread and Milk occur together in 11 of the 20 transactions.

- Bread → Milk has confidence `11/14 = 0.7857`.
- Milk → Bread has confidence `11/15 = 0.7333`.
- Both rules have lift **1.0476**.

Their association is positive but close to the independence baseline of 1.

#### What the New Dataset Shows

All five selected rules have lift above 1. Butter and Bread have the highest lift and could be considered for a joint promotion.

However, these results come from a small synthetic dataset. Actual customer data would be needed before making business decisions or generalising the findings.

---

### 3. Effect of Changing the Algorithm and Thresholds

#### Apriori and FP-Growth

Apriori and FP-Growth produce the same frequent itemsets and association-rule metrics when applied to the same dataset with the same thresholds.

Apriori generates and tests candidate itemsets. FP-Growth uses an FP-tree and avoids explicit candidate generation.

With minimum support **0.5** and minimum confidence **0.7**, both algorithms produce **6 frequent itemsets and 5 rules** for the new dataset.

This experiment checks their results. It does not establish which algorithm runs faster.

#### Threshold Comparison

The following results were obtained for both algorithms using the same 20 transactions.

| Scenario | Minimum Support | Minimum Confidence | Frequent Itemsets | Rules |
|:---|---:|---:|---:|---:|
| Baseline | 0.5 | 0.7 | 6 | 5 |
| Lower support | 0.3 | 0.7 | 11 | 9 |
| Higher support | 0.6 | 0.7 | 3 | 0 |
| Lower confidence | 0.5 | 0.5 | 6 | 6 |
| Higher confidence | 0.5 | 0.9 | 6 | 1 |
| Both lower | 0.3 | 0.5 | 11 | 14 |

#### Lowering Minimum Support

Reducing support from **0.5 to 0.3** allows itemsets appearing in at least **6 transactions**, instead of 10.

The number of frequent itemsets increases from **6 to 11**, and the number of rules increases from **5 to 9**.

For example, the rule **Bread and Milk → Butter** becomes available:

- Support = `9/20 = 0.45`.
- Confidence = `9/11 = 0.8182`.
- Lift = **1.3636**.

This rule was excluded at support 0.5 because its support is only 0.45.

#### Increasing Minimum Support

Increasing support to **0.6** requires an itemset to appear in at least **12 transactions**.

Only the single items Bread, Butter and Milk qualify. No pair appears in at least 12 transactions, so the result contains **3 frequent itemsets and 0 rules**.

An output with no rules is valid when no combinations meet the selected thresholds.

#### Lowering Minimum Confidence

Reducing confidence from **0.7 to 0.5** keeps the same **6 frequent itemsets**, but increases the number of rules from **5 to 6**.

The additional rule is **Milk → Butter**:

- Support = **0.5000**.
- Confidence = `10/15 = 0.6667`.
- Lift = **1.1111**.

Its confidence is below 0.7 but above 0.5, so it becomes eligible at the lower threshold.

#### Increasing Minimum Confidence

Increasing confidence to **0.9** keeps the same **6 frequent itemsets**, but reduces the number of rules from **5 to 1**.

Only **Butter → Bread** remains because its confidence is approximately **0.9167**. All other rules have confidence below 0.9.

#### Lowering Both Thresholds

Using minimum support **0.3** and minimum confidence **0.5** produces **11 frequent itemsets and 14 rules**.

This is the largest rule set among the settings tested. However, more rules do not automatically mean more useful results. The added rules include less frequent or lower-confidence patterns.

Changing thresholds changes which itemsets and rules are selected. It does not change the support, confidence or lift of a particular rule when the dataset stays the same.